
# Milvus on Zilliz Cloud

This notebook shows **how to connect directly to a cloud‑hosted Milvus database (Zilliz Cloud)** and perform:

- Connection & authentication
- Collection inspection
- CRUD operations
- Vector similarity search
- Visual exploration of embeddings (2D)

Designed for **teaching Milvus concepts**, not local Docker setups.



## 1 Install & Import Dependencies


In [ ]:
%pip install pymilvus


## 2 Connect to Zilliz Cloud (Milvus)



In [19]:
!python --version

Python 3.13.5


In [ ]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")



## 3 Inspect Collections


In [ ]:

from pymilvus import utility

utility.list_collections()



## 4 Load & Inspect a Collection


In [26]:

from pymilvus import Collection

# Collection is same as a Table in traditional databases
collection = Collection("demo_collection")
collection.load()

collection.schema


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_73754/1253288197.py:4: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection("demo_collection")
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_73754/1253288197.py:5: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()


{'auto_id': True, 'description': 'demo_collection', 'fields': [{'name': 'id', 'description': 'The Primary Key', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 4}}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}], 'enable_dynamic_field': True, 'enable_namespace': False}


## 5 Read Data (Query)

Query a few rows to see **raw stored data**


In [ ]:
results = collection.query(
    expr="title like '%AI%'",  
    # output_fields=["id", "title", "vector"],
    output_fields=["primary_key", "title"],
    limit=5
)
# select id, title, vector from demo_collection where id >= 0 limit 5
# select id, title from demo_collection where id >= 0 limit 5
# select <output_fields> from <Collection> where <expression> limit <number>
results


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_73754/1547826186.py:1: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.query(


data: ["{'primary_key': 20}"], extra_info: {'cost': 6, 'scanned_remote_bytes': 0, 'scanned_total_bytes': 3145728, 'cache_hit_ratio': 1.0}

In [22]:
results = collection.query(
    expr="id in [463705163763347400, 463705164234735154] AND title == 'Deep Learning (Updated)'",
    
    output_fields=["id", "title", "vector"],
    limit=5
)
# select id, title, vector from demo_collection 
# where id in [463705163763347400, 463705164234735154] AND title == 'Deep Learning (Updated)' limit 5
results


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_73754/2831894744.py:1: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.query(


data: [], extra_info: {'cost': 6, 'scanned_remote_bytes': 0, 'scanned_total_bytes': 2097152, 'cache_hit_ratio': 1.0}


## 6 Insert (CREATE)

Insert a new vector record


In [ ]:
import numpy as np

data = [
    [[.3456, .2345, .1234, .5678]],            # vector FIRST
    ["Milvus makes vector search scalable - May 27th 2026"]     # title SECOND
]

collection.insert(data)
collection.flush()



## 7 Update (DELETE + INSERT pattern)

Milvus does not support in‑place updates.


In [ ]:
# collection.delete(expr="id == 463705163763347399")
# collection.flush()

updated_data = [
    [463705165204270561],  # ← list of IDs (1 row)
    [[0.7000895, 0.022113776, 0.48144588, 0.23203984]],  # ← list of vectors (1 row)
    ["Next.JS for Beginners - Updated May 27th 2026"]   # ← list of titles (1 row)
]

result = collection.upsert(updated_data)
collection.flush()

new_id = result.primary_keys[0]
print(f"Record updated. New ID generated: {new_id}")



## 8 Delete


In [ ]:

collection.delete(expr="id like '466305617795297852")
collection.flush()

print("Record deleted")



/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_73754/2220947547.py:1: PyMilvusDeprecationWarning: `Collection.delete` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.delete(expr="id like '46630561779529785%'")
2026-05-27 20:06:10,124 [ERROR][_log_rpc_error]: RPC error: [delete], <MilvusException: (code=1100, message=failed to create delete plan: cannot parse expression: id like '46630561779529785%', error: like operation on non-string or no-json field is unsupported: invalid parameter)>, <elapsed:652.7ms>
Traceback:
Traceback (most recent call last):
  File "/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/pymilvus/decorators.py", line 518, in handler
    return func(*args, **kwargs)
  File "/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/pymilvus/decorators.py", line 565, in handler
    return func(self, *args, **kwargs)
  File "/Users

MilvusException: <MilvusException: (code=1100, message=failed to create delete plan: cannot parse expression: id like '46630561779529785%', error: like operation on non-string or no-json field is unsupported: invalid parameter)>

In [ ]:

results = collection.query(
    expr="id == 463705164234754808",
    output_fields=[ "title", "vector"],
    # output_fields=["*"],
    limit=5
)

results



## 9 Vector Similarity Search


In [ ]:
query_vector = [0.4670895, 0.343513776, 0.22224588, 0.113984]
print(f"Query Vector: {query_vector}")
search_results = collection.search(
    data=[query_vector],
    anns_field="vector",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=2,
    output_fields=["id","title"]
)
# SELECT id, title FROM demo_collection WHERE COSINE_SIMILARITY(vector, [0.4670895, 0.343513776, 0.22224588, 0.113984]) > threshold LIMIT 2
print(f"Search Results: {search_results}")

for hit in search_results[0]:
    print(f"title={hit.entity.get('title')}")
